<a href="https://colab.research.google.com/github/josephgalicinao/SkinLesionDetection/blob/main/Skin_Lesion_CNNs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import nbformat

path = "/content/drive/MyDrive/Colab Notebooks/Skin Lesion - CNNs.ipynb"

nb = nbformat.read(path, as_version=4)

if "widgets" in nb.metadata:
    del nb.metadata["widgets"]

nbformat.write(nb, path)
print(f"Cleaned {path}")

Cleaned /content/drive/MyDrive/Colab Notebooks/Skin Lesion - CNNs.ipynb


## Imports

Important imports and packages used

In [ ]:
from google.colab import userdata
import os

import kagglehub
import pandas as pd
import tensorflow as tf
import os
import csv
import numpy as np
import matplotlib.pyplot as plt
import cv2
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedGroupKFold

from sklearn.model_selection import cross_val_score
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
import csv
import pandas as pd
from sklearn.preprocessing import StandardScaler
from imblearn.combine import SMOTETomek
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from mlxtend.classifier import StackingClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from tqdm import tqdm
from sklearn.metrics import roc_auc_score
from sklearn.metrics import precision_recall_curve, auc

import torch
import transformers
import torchvision.transforms as T
from transformers import ViTForImageClassification, ViTImageProcessor, AutoConfig
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
from PIL import Image
from transformers import Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from transformers import AutoModelForImageClassification, AutoImageProcessor

# import selectkbest
from sklearn.feature_selection import SelectKBest, f_classif

IMG_SIZE   = (128, 128)
AUTOTUNE   = tf.data.AUTOTUNE

label_map = {'Benign': 0,
             'Malignant': 1}

## Load Dataset

In [ ]:
os.environ["KAGGLE_USERNAME"] =  userdata.get('KAGGLE_USERNAME')
os.environ["KAGGLE_KEY"] = userdata.get('KAGGLE_KEY')

!pip install -q kaggle

ISIC 2018

In [ ]:
isic2018_path = kagglehub.dataset_download("josephgalicinao/isic-2018-dataset")

print("Path to dataset files:", isic2018_path)

100%|██████████| 237M/237M [00:17<00:00, 13.9MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/josephgalicinao/isic-2018-dataset/versions/1


ISIC 2018 Test

In [ ]:
# Download latest version
isic2018_test_path = kagglehub.dataset_download("josephgalicinao/isic-2018-test-dataset")

print("Path to dataset files:", isic2018_test_path)

100%|██████████| 37.3M/37.3M [00:03<00:00, 10.2MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/josephgalicinao/isic-2018-test-dataset/versions/1


Get the metadata of the datasets

In [ ]:
def get_train_isic2018():
  print("ISIC 2018 Training Dataset...")
  df = pd.read_csv(f"{isic2018_path}/metadata.csv")
  df = df.dropna(subset=['diagnosis_1'])

  isic_ids = df["isic_id"].values
  isic_dx = df["diagnosis_1"].map({'Malignant': 1, 'Benign': 0, 'Indeterminate': 1}).values
  isic_patient_ids = df["lesion_id"].values

  isic_paths = []
  for isic_id in isic_ids:
    isic_paths.append(f"{isic2018_path}/{isic_id}.jpg")

  assert len(isic_paths) == len(isic_dx)

  return isic_paths, isic_dx, isic_patient_ids

def get_test_isic2018():
  print("ISIC 2018 Test Dataset...")
  df = pd.read_csv(f"{isic2018_test_path}/metadata.csv")
  df = df.dropna(subset=['diagnosis_1'])

  isic_ids = df["isic_id"].values
  isic_dx = df["diagnosis_1"].map({'Malignant': 1, 'Benign': 0, 'Indeterminate': 1}).values

  isic_paths = []
  for isic_id in isic_ids:
    isic_paths.append(f"{isic2018_test_path}/{isic_id}.jpg")

  assert len(isic_paths) == len(isic_dx)

  return isic_paths, isic_dx

Make the straifier and k fold

In [ ]:
# Stratifier
sgkf = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

## Evaluation Metrics

In [ ]:
def get_metrics(y_val, y_pred):
  # y_true: true labels, y_scores: predicted probabilities
  precision, recall, thresholds = precision_recall_curve(y_val, y_pred)
  pr_auc = auc(recall, precision)
  precision = precision_score(y_val, y_pred, average='binary')
  recall = recall_score(y_val, y_pred, average='binary')
  f1 = f1_score(y_val, y_pred, average='binary')

  return [precision, recall, f1, pr_auc]

# CNN-Based Transfer Learning

## Dataset

In [ ]:
# Dataset class
class SkinLesionDataset(Dataset):
    def __init__(self, labels, img_paths, transform=None):
        self.labels = labels
        self.img_paths = img_paths
        self.transform = transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        image = Image.open(self.img_paths[idx]).convert("RGB")
        label = torch.tensor(self.labels[idx], dtype=torch.long)

        if self.transform is not None:
            image = self.transform(image)

        return {
            "pixel_values": image,
            "labels": label,
        }

## Metrics

In [ ]:
!pip install evaluate
import evaluate
import numpy as np
from sklearn.metrics import average_precision_score

accuracy = evaluate.load("accuracy")
precision = evaluate.load("precision")
recall = evaluate.load("recall")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    probs = np.exp(logits) / np.exp(logits).sum(-1, keepdims=True)

    metrics = {}
    metrics.update(accuracy.compute(predictions=preds, references=labels))
    metrics.update(precision.compute(predictions=preds, references=labels, average="binary"))
    metrics.update(recall.compute(predictions=preds, references=labels, average="binary"))
    metrics.update(f1.compute(predictions=preds, references=labels, average="binary"))

    metrics["pr_auc"] = average_precision_score(labels, probs[:, 1])

    return metrics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.2 MB/s eta 0:00:00


## Preprocess

In [ ]:
def preprocess(image_processor):
  train_transform = T.Compose([
      T.Resize((224, 224)),
      T.RandomHorizontalFlip(p=0.5),
      T.RandomVerticalFlip(p=0.5),
      T.RandomApply([
          T.ColorJitter(
              brightness=0.1,
              contrast=0.1,
              saturation=0.1,
              hue=0.05
          )
      ], p=0.5),
      T.ToTensor(),
      T.Normalize(mean=image_processor.image_mean, std=image_processor.image_std),
  ])

  val_transform = T.Compose([
      T.Resize((224, 224)),
      T.ToTensor(),
      T.Normalize(mean=image_processor.image_mean, std=image_processor.image_std),
  ])

  return train_transform, val_transform

## ResNet50

In [ ]:
model_name = "microsoft/resnet-50"

img_paths, dx, patient_ids = get_train_isic2018()
img_paths = np.array(img_paths)
dx = np.array(dx)

with open('/content/drive/MyDrive/Thesis/isic2018_fine_resnet50.csv', 'w', newline='') as csvfile:
  writer = csv.writer(csvfile)

  # header
  writer.writerow(["Unfrozen Layers", "Accuracy", "Precision", "Recall", "F1", "PR-AUC"])

  for prop in [0.75, 0.5, 0.25, 0]:
    print(f"Prop: {prop}")
    for fold, (train_idx, val_idx) in enumerate(sgkf.split(img_paths, dx, patient_ids)):
      train_imgs, val_imgs = img_paths[train_idx], img_paths[val_idx]
      train_dx, val_dx = dx[train_idx], dx[val_idx]
      train_imgs, train_dx, _, _ = oversample_minority(train_imgs, train_dx)

      # Get Preprocessed Data
      image_processor = AutoImageProcessor.from_pretrained(model_name)
      train_transform, val_transform = preprocess(image_processor)
      train_ds = SkinLesionDataset(train_dx, train_imgs, transform=train_transform)
      val_ds = SkinLesionDataset(val_dx, val_imgs, transform=val_transform)

      model = AutoModelForImageClassification.from_pretrained(
          model_name,
          num_labels=2,
          ignore_mismatched_sizes=True
      )

      # Replace classification head
      print(model.classifier)
      in_features = model.classifier[1].in_features

      model.classifier = torch.nn.Sequential(
          torch.nn.Flatten(),
          torch.nn.Linear(in_features, 128),
          torch.nn.ReLU(),
          torch.nn.Linear(128, 2)
      )

      # Freeze backbone
      for param in model.resnet.parameters():
          param.requires_grad = False

      # Collect ResNet stages
      stages = list(model.resnet.encoder.stages)

      # Unfreeze last proportion of stages
      n_stages = len(stages)
      n_unfreeze = max(1, int(n_stages * prop))  # avoid 0
      print(f"Unfrozen Stages: {n_unfreeze} / {n_stages}")

      for stage in stages[-n_unfreeze:]:
          for param in stage.parameters():
              param.requires_grad = True

      # Always train classifier
      for param in model.classifier.parameters():
          param.requires_grad = True

      # Train the model
      training_args = TrainingArguments(
          output_dir="./trains",
          per_device_train_batch_size=64,
          per_device_eval_batch_size=64,
          eval_strategy="epoch",
          save_strategy="epoch",
          logging_strategy="epoch",
          num_train_epochs=10,
          learning_rate=1e-5,
          save_total_limit=2,
          remove_unused_columns=False,
          load_best_model_at_end=True,
          metric_for_best_model="pr_auc",
          greater_is_better=True,
          dataloader_num_workers=8,
          dataloader_pin_memory=True,
          fp16=torch.cuda.is_available(),
          report_to="none",
          disable_tqdm=False,
      )

      trainer = Trainer(
          model=model,
          args=training_args,
          train_dataset=train_ds,
          eval_dataset=val_ds,
          compute_metrics=compute_metrics
      )

      trainer.train()

      eval_results = trainer.evaluate()

      print(eval_results)

      writer.writerow([
          prop,
          eval_results.get("eval_accuracy"),
          eval_results.get("eval_precision"),
          eval_results.get("eval_recall"),
          eval_results.get("eval_f1"),
          eval_results.get("eval_pr_auc"),
      ])

ISIC 2018 Training Dataset...
Prop: 0.75


preprocessor_config.json:   0%|          | 0.00/266 [00:00<?, ?B/s]

The image processor of type `ConvNextImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/102M [00:00<?, ?B/s]

...

ResNetForImageClassification LOAD REPORT from: microsoft/resnet-50
Key                 | Status   |                                                                                          
--------------------+----------+------------------------------------------------------------------------------------------
classifier.1.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000, 2048]) vs model:torch.Size([2, 2048])
classifier.1.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000]) vs model:torch.Size([2])            

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


Sequential(
  (0): Flatten(start_dim=1, end_dim=-1)
  (1): Linear(in_features=2048, out_features=2, bias=True)
)
Unfrozen Stages: 3 / 4


Epoch,Training Loss,Validation Loss


Exception in thread Thread-26 (_pin_memory_loop):
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/pin_memory.py", line 52, in _pin_memory_loop
    do_one_step()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/pin_memory.py", line 28, in do_one_step
    r = in_queue.get(timeout=MP_STATUS_CHECK_INTERVAL)
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/queues.py", line 122, in get
    return _ForkingPickler.loads(res)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/multiprocessing/reductions.py", line 540, in rebuild_storage_fd
    fd = df.detach()
         ^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/resource_

KeyboardInterrupt: 

Evaluate

In [ ]:
model_name = "microsoft/resnet-50"

prop = 0.75

train_imgs, train_dx, _ = get_train_isic2018()
train_imgs, train_dx, _, _ = oversample_minority(train_imgs, train_dx)
val_imgs, val_dx = get_test_isic2018()

with open('/content/drive/MyDrive/Thesis/isic2018_test_resnet50.csv', 'w', newline='') as csvfile:
  writer = csv.writer(csvfile)

  # header
  writer.writerow(["Unfrozen Layers", "Accuracy", "Precision", "Recall", "F1", "PR-AUC"])

  # Get Preprocessed Data
  image_processor = AutoImageProcessor.from_pretrained(model_name)
  train_transform, val_transform = preprocess(image_processor)
  train_ds = SkinLesionDataset(train_dx, train_imgs, transform=train_transform)
  val_ds = SkinLesionDataset(val_dx, val_imgs, transform=val_transform)

  model = AutoModelForImageClassification.from_pretrained(
      model_name,
      num_labels=2,
      ignore_mismatched_sizes=True
  )

  # Replace classification head
  print(model.classifier)
  in_features = model.classifier[1].in_features

  model.classifier = torch.nn.Sequential(
      torch.nn.Flatten(),
      torch.nn.Linear(in_features, 128),
      torch.nn.ReLU(),
      torch.nn.Linear(128, 2)
  )

  # Freeze backbone
  for param in model.resnet.parameters():
      param.requires_grad = False

  # Collect ResNet stages
  stages = list(model.resnet.encoder.stages)

  # Unfreeze last proportion of stages
  n_stages = len(stages)
  n_unfreeze = max(1, int(n_stages * prop))  # avoid 0
  print(f"Unfrozen Stages: {n_unfreeze} / {n_stages}")

  for stage in stages[-n_unfreeze:]:
      for param in stage.parameters():
          param.requires_grad = True

  # Always train classifier
  for param in model.classifier.parameters():
      param.requires_grad = True

  # Train the model
  training_args = TrainingArguments(
      output_dir="./trains",
      per_device_train_batch_size=64,
      per_device_eval_batch_size=64,
      eval_strategy="epoch",
      save_strategy="epoch",
      logging_strategy="epoch",
      num_train_epochs=10,
      learning_rate=1e-5,
      save_total_limit=2,
      remove_unused_columns=False,
      load_best_model_at_end=True,
      metric_for_best_model="pr_auc",
      greater_is_better=True,
      dataloader_num_workers=8,
      dataloader_pin_memory=True,
      fp16=torch.cuda.is_available(),
      report_to="none",
      disable_tqdm=False,
  )

  trainer = Trainer(
      model=model,
      args=training_args,
      train_dataset=train_ds,
      eval_dataset=val_ds,
      compute_metrics=compute_metrics
  )

  trainer.train()

  eval_results = trainer.evaluate()

  print(eval_results)

  writer.writerow([
      prop,
      eval_results.get("eval_accuracy"),
      eval_results.get("eval_precision"),
      eval_results.get("eval_recall"),
      eval_results.get("eval_f1"),
      eval_results.get("eval_pr_auc"),
  ])

ISIC 2018 Training Dataset...
ISIC 2018 Test Dataset...


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

ResNetForImageClassification LOAD REPORT from: microsoft/resnet-50
Key                 | Status   |                                                                                          
--------------------+----------+------------------------------------------------------------------------------------------
classifier.1.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000, 2048]) vs model:torch.Size([2, 2048])
classifier.1.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000]) vs model:torch.Size([2])            

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


Sequential(
  (0): Flatten(start_dim=1, end_dim=-1)
  (1): Linear(in_features=2048, out_features=2, bias=True)
)
Unfrozen Stages: 3 / 4


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Pr Auc
1,0.671351,0.664244,0.658730,0.360107,0.876221,0.510436,0.473106
2,0.580284,0.556873,0.708995,0.400300,0.869707,0.548255,0.530878
3,0.487768,0.511159,0.713624,0.405405,0.879479,0.554985,0.571106
4,0.448769,0.506348,0.715608,0.408346,0.892508,0.560327,0.610869
5,0.431393,0.490149,0.724206,0.416159,0.889251,0.566978,0.620120
6,0.423798,0.482900,0.733466,0.424765,0.882736,0.573545,0.632913
7,0.414490,0.473766,0.743386,0.434572,0.876221,0.580994,0.635825
8,0.408504,0.486288,0.728175,0.420245,0.892508,0.571429,0.648485
9,0.406359,0.465044,0.746032,0.436782,0.866450,0.580786,0.649774
10,0.409066,0.489758,0.734127,0.426584,0.899023,0.578616,0.649485


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'eval_loss': 0.4650435745716095, 'eval_accuracy': 0.746031746031746, 'eval_precision': 0.4367816091954023, 'eval_recall': 0.8664495114006515, 'eval_f1': 0.5807860262008734, 'eval_pr_auc': 0.6497735982268769, 'eval_runtime': 2.267, 'eval_samples_per_second': 666.964, 'eval_steps_per_second': 10.587, 'epoch': 10.0}


## MobileNetV2

In [ ]:
model_name = "google/mobilenet_v2_1.4_224"

img_paths, dx, patient_ids = get_train_isic2018()
img_paths = np.array(img_paths)
dx = np.array(dx)

with open('/content/drive/MyDrive/Thesis/isic2018_fine_mobilenetv2.csv', 'w', newline='') as csvfile:
  writer = csv.writer(csvfile)

  # header
  writer.writerow(["Unfrozen Layers", "Accuracy", "Precision", "Recall", "F1", "PR-AUC"])

  for prop in [0.75, 0.5, 0.25, 0]:
    print(f"Prop: {prop}")
    for fold, (train_idx, val_idx) in enumerate(sgkf.split(img_paths, dx, patient_ids)):
      train_imgs, val_imgs = img_paths[train_idx], img_paths[val_idx]
      train_dx, val_dx = dx[train_idx], dx[val_idx]
      train_imgs, train_dx, _, _ = oversample_minority(train_imgs, train_dx)

      # Get Preprocessed Data
      image_processor = AutoImageProcessor.from_pretrained(model_name)
      train_transform, val_transform = preprocess(image_processor)
      train_ds = SkinLesionDataset(train_dx, train_imgs, transform=train_transform)
      val_ds = SkinLesionDataset(val_dx, val_imgs, transform=val_transform)

      model = AutoModelForImageClassification.from_pretrained(
          model_name,
          num_labels=2,
          ignore_mismatched_sizes=True
      )

      # Replace classification head
      print(model.classifier)
      in_features = model.classifier.in_features

      model.classifier = torch.nn.Sequential(
          torch.nn.Flatten(),
          torch.nn.Linear(in_features, 128),
          torch.nn.ReLU(),
          torch.nn.Linear(128, 2)
      )

      # Freeze backbone
      for param in model.base_model.parameters():
        param.requires_grad = False

      # Collect MobileNetV2 stages
      blocks = list(model.base_model.layer)

      n_blocks = len(blocks)
      n_unfreeze = max(1, int(n_blocks * prop))

      print(f"Unfrozen Blocks: {n_unfreeze} / {n_blocks}")

      for block in blocks[-n_unfreeze:]:
          for param in block.parameters():
              param.requires_grad = True

      # Always train classifier
      for param in model.classifier.parameters():
          param.requires_grad = True

      # Train the model
      training_args = TrainingArguments(
          output_dir="./trains",
          per_device_train_batch_size=64,
          per_device_eval_batch_size=64,
          eval_strategy="epoch",
          save_strategy="epoch",
          logging_strategy="epoch",
          num_train_epochs=10,
          learning_rate=1e-5,
          save_total_limit=2,
          remove_unused_columns=False,
          load_best_model_at_end=True,
          metric_for_best_model="pr_auc",
          greater_is_better=True,
          dataloader_num_workers=8,
          dataloader_pin_memory=True,
          fp16=torch.cuda.is_available(),
          report_to="none",
          disable_tqdm=False,
      )

      trainer = Trainer(
          model=model,
          args=training_args,
          train_dataset=train_ds,
          eval_dataset=val_ds,
          compute_metrics=compute_metrics
      )

      trainer.train()

      eval_results = trainer.evaluate()

      print(eval_results)

      writer.writerow([
          prop,
          eval_results.get("eval_accuracy"),
          eval_results.get("eval_precision"),
          eval_results.get("eval_recall"),
          eval_results.get("eval_f1"),
          eval_results.get("eval_pr_auc"),
      ])

Evaluation

In [ ]:
model_name = "google/mobilenet_v2_1.4_224"

prop = 0.75
train_imgs, train_dx, _ = get_train_isic2018()
train_imgs, train_dx, _, _ = oversample_minority(train_imgs, train_dx)
val_imgs, val_dx = get_test_isic2018()

with open('/content/drive/MyDrive/Thesis/isic2018_fine_mobilenetv2.csv', 'w', newline='') as csvfile:
  writer = csv.writer(csvfile)

  # header
  writer.writerow(["Unfrozen Layers", "Accuracy", "Precision", "Recall", "F1", "PR-AUC"])

  # Get Preprocessed Data
  image_processor = AutoImageProcessor.from_pretrained(model_name)
  train_transform, val_transform = preprocess(image_processor)
  train_ds = SkinLesionDataset(train_dx, train_imgs, transform=train_transform)
  val_ds = SkinLesionDataset(val_dx, val_imgs, transform=val_transform)

  model = AutoModelForImageClassification.from_pretrained(
      model_name,
      num_labels=2,
      ignore_mismatched_sizes=True
  )

  # Replace classification head
  print(model.classifier)
  in_features = model.classifier.in_features

  model.classifier = torch.nn.Sequential(
      torch.nn.Flatten(),
      torch.nn.Linear(in_features, 128),
      torch.nn.ReLU(),
      torch.nn.Linear(128, 2)
  )

  # Freeze backbone
  for param in model.base_model.parameters():
    param.requires_grad = False

  # Collect MobileNetV2 stages
  blocks = list(model.base_model.layer)

  n_blocks = len(blocks)
  n_unfreeze = max(1, int(n_blocks * prop))

  print(f"Unfrozen Blocks: {n_unfreeze} / {n_blocks}")

  for block in blocks[-n_unfreeze:]:
      for param in block.parameters():
          param.requires_grad = True

  # Always train classifier
  for param in model.classifier.parameters():
      param.requires_grad = True

  # Train the model
  training_args = TrainingArguments(
      output_dir="./trains",
      per_device_train_batch_size=64,
      per_device_eval_batch_size=64,
      eval_strategy="epoch",
      save_strategy="epoch",
      logging_strategy="epoch",
      num_train_epochs=10,
      learning_rate=1e-5,
      save_total_limit=2,
      remove_unused_columns=False,
      load_best_model_at_end=True,
      metric_for_best_model="pr_auc",
      greater_is_better=True,
      dataloader_num_workers=8,
      dataloader_pin_memory=True,
      fp16=torch.cuda.is_available(),
      report_to="none",
      disable_tqdm=False,
  )

  trainer = Trainer(
      model=model,
      args=training_args,
      train_dataset=train_ds,
      eval_dataset=val_ds,
      compute_metrics=compute_metrics
  )

  trainer.train()

  eval_results = trainer.evaluate()

  print(eval_results)

  writer.writerow([
      prop,
      eval_results.get("eval_accuracy"),
      eval_results.get("eval_precision"),
      eval_results.get("eval_recall"),
      eval_results.get("eval_f1"),
      eval_results.get("eval_pr_auc"),
  ])

ISIC 2018 Training Dataset...
ISIC 2018 Test Dataset...


preprocessor_config.json:   0%|          | 0.00/406 [00:00<?, ?B/s]

The image processor of type `MobileNetV2ImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/24.7M [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'eval_loss': 0.35803791880607605, 'eval_accuracy': 0.8333333333333334, 'eval_precision': 0.5638051044083526, 'eval_recall': 0.7915309446254072, 'eval_f1': 0.6585365853658537, 'eval_pr_auc': 0.7462038203696529, 'eval_runtime': 2.1284, 'eval_samples_per_second': 710.384, 'eval_steps_per_second': 11.276, 'epoch': 10.0}


## ConvNeXtTiny

In [ ]:
model_name = "facebook/convnext-tiny-224"

img_paths, dx, patient_ids = get_train_isic2018()
img_paths = np.array(img_paths)
dx = np.array(dx)

with open('/content/drive/MyDrive/Thesis/isic2018_fine_convnexttiny.csv', 'w', newline='') as csvfile:
  writer = csv.writer(csvfile)

  # header
  writer.writerow(["Unfrozen Layers", "Accuracy", "Precision", "Recall", "F1", "PR-AUC"])

  for prop in [0.75, 0.5, 0.25, 0]:
    print(f"Prop: {prop}")
    for fold, (train_idx, val_idx) in enumerate(sgkf.split(img_paths, dx, patient_ids)):
      train_imgs, val_imgs = img_paths[train_idx], img_paths[val_idx]
      train_dx, val_dx = dx[train_idx], dx[val_idx]
      train_imgs, train_dx, _, _ = oversample_minority(train_imgs, train_dx)

      # Get Preprocessed Data
      image_processor = AutoImageProcessor.from_pretrained(model_name)
      train_transform, val_transform = preprocess(image_processor)
      train_ds = SkinLesionDataset(train_dx, train_imgs, transform=train_transform)
      val_ds = SkinLesionDataset(val_dx, val_imgs, transform=val_transform)

      model = AutoModelForImageClassification.from_pretrained(
          model_name,
          num_labels=2,
          ignore_mismatched_sizes=True
      )

      # Replace classification head
      print(model.classifier)
      in_features = model.classifier.in_features

      model.classifier = torch.nn.Sequential(
          torch.nn.Flatten(),
          torch.nn.Linear(in_features, 128),
          torch.nn.ReLU(),
          torch.nn.Linear(128, 2)
      )

      # Freeze backbone
      for param in model.convnext.parameters():
        param.requires_grad = False

      # Collect MobileNetV2 stages
      blocks = list(model.convnext.encoder.stages)

      n_blocks = len(blocks)
      n_unfreeze = max(1, int(n_blocks * prop))

      print(f"Unfrozen Blocks: {n_unfreeze} / {n_blocks}")

      for block in blocks[-n_unfreeze:]:
          for param in block.parameters():
              param.requires_grad = True

      # Always train classifier
      for param in model.classifier.parameters():
          param.requires_grad = True

      # Train the model
      training_args = TrainingArguments(
          output_dir="./trains",
          per_device_train_batch_size=64,
          per_device_eval_batch_size=64,
          eval_strategy="epoch",
          save_strategy="epoch",
          logging_strategy="epoch",
          num_train_epochs=10,
          learning_rate=1e-5,
          save_total_limit=2,
          remove_unused_columns=False,
          load_best_model_at_end=True,
          metric_for_best_model="pr_auc",
          greater_is_better=True,
          dataloader_num_workers=8,
          dataloader_pin_memory=True,
          fp16=torch.cuda.is_available(),
          report_to="none",
          disable_tqdm=False,
      )

      trainer = Trainer(
          model=model,
          args=training_args,
          train_dataset=train_ds,
          eval_dataset=val_ds,
          compute_metrics=compute_metrics
      )

      trainer.train()

      eval_results = trainer.evaluate()

      print(eval_results)

      writer.writerow([
          prop,
          eval_results.get("eval_accuracy"),
          eval_results.get("eval_precision"),
          eval_results.get("eval_recall"),
          eval_results.get("eval_f1"),
          eval_results.get("eval_pr_auc"),
      ])

Evaluate

In [ ]:
model_name = "facebook/convnext-tiny-224"

prop = 0.75
train_imgs, train_dx, _ = get_train_isic2018()
train_imgs, train_dx, _, _ = oversample_minority(train_imgs, train_dx)
val_imgs, val_dx = get_test_isic2018()

with open('/content/drive/MyDrive/Thesis/isic2018_fine_convnexttiny.csv', 'w', newline='') as csvfile:
  writer = csv.writer(csvfile)

  # header
  writer.writerow(["Unfrozen Layers", "Accuracy", "Precision", "Recall", "F1", "PR-AUC"])

  # Get Preprocessed Data
  image_processor = AutoImageProcessor.from_pretrained(model_name)
  train_transform, val_transform = preprocess(image_processor)
  train_ds = SkinLesionDataset(train_dx, train_imgs, transform=train_transform)
  val_ds = SkinLesionDataset(val_dx, val_imgs, transform=val_transform)

  model = AutoModelForImageClassification.from_pretrained(
      model_name,
      num_labels=2,
      ignore_mismatched_sizes=True
  )

  # Replace classification head
  print(model.classifier)
  in_features = model.classifier.in_features

  model.classifier = torch.nn.Sequential(
      torch.nn.Flatten(),
      torch.nn.Linear(in_features, 128),
      torch.nn.ReLU(),
      torch.nn.Linear(128, 2)
  )

  # Freeze backbone
  for param in model.convnext.parameters():
    param.requires_grad = False

  # Collect MobileNetV2 stages
  blocks = list(model.convnext.encoder.stages)

  n_blocks = len(blocks)
  n_unfreeze = max(1, int(n_blocks * prop))

  print(f"Unfrozen Blocks: {n_unfreeze} / {n_blocks}")

  for block in blocks[-n_unfreeze:]:
      for param in block.parameters():
          param.requires_grad = True

  # Always train classifier
  for param in model.classifier.parameters():
      param.requires_grad = True

  # Train the model
  training_args = TrainingArguments(
      output_dir="./trains",
      per_device_train_batch_size=64,
      per_device_eval_batch_size=64,
      eval_strategy="epoch",
      save_strategy="epoch",
      logging_strategy="epoch",
      num_train_epochs=10,
      learning_rate=1e-5,
      save_total_limit=2,
      remove_unused_columns=False,
      load_best_model_at_end=True,
      metric_for_best_model="pr_auc",
      greater_is_better=True,
      dataloader_num_workers=8,
      dataloader_pin_memory=True,
      fp16=torch.cuda.is_available(),
      report_to="none",
      disable_tqdm=False,
  )

  trainer = Trainer(
      model=model,
      args=training_args,
      train_dataset=train_ds,
      eval_dataset=val_ds,
      compute_metrics=compute_metrics
  )

  trainer.train()

  eval_results = trainer.evaluate()

  print(eval_results)

  writer.writerow([
      prop,
      eval_results.get("eval_accuracy"),
      eval_results.get("eval_precision"),
      eval_results.get("eval_recall"),
      eval_results.get("eval_f1"),
      eval_results.get("eval_pr_auc"),
  ])

ISIC 2018 Training Dataset...
ISIC 2018 Test Dataset...


preprocessor_config.json:   0%|          | 0.00/266 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/114M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/182 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'eval_loss': 0.35761919617652893, 'eval_accuracy': 0.8425925925925926, 'eval_precision': 0.5819477434679335, 'eval_recall': 0.7980456026058632, 'eval_f1': 0.6730769230769231, 'eval_pr_auc': 0.7618640738145215, 'eval_runtime': 2.3772, 'eval_samples_per_second': 636.036, 'eval_steps_per_second': 10.096, 'epoch': 10.0}
